# Step 00 — Data provenance and objective reproducibility audit

This notebook reruns the **current step 00 contract** with two explicit goals:

1. keep a hard separation between the **legacy historical Optuna DB/CSV assets** and the **new ATF dataset**;
2. make the remaining unresolved historical-control provenance explicit rather than trying to over-interpret it.

## Reuse from `astro_atf_analysis_improved_sectioned.ipynb`

This notebook does **not** reuse the ATF signal preprocessing or feature extraction logic from the reference notebook.  
It only uses the same ATF filename contract (`DH`/`VH`, `CONTROL`/`MFA`/`MFA_BA`) so that step 00 and step 02 stay consistent about what the ATF files are.

## Important interpretation change

`CONTROL_TRACES_old.csv` has been removed and is not used here.  
That cleanup is correct, but it does **not** validate the historical control DB objectives: with the documented `CONTROL_TRACES.csv` source, the historical control fits remain unresolved.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", Path.cwd())).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.provenance import run_step00_provenance

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 200)

PROJECT_ROOT

In [ ]:
results = run_step00_provenance(PROJECT_ROOT)

db_summary = results["db_study_summary"]
trace_summary = results["trace_source_summary"]
provenance = results["control_trace_verification"]
atf_inventory = results["atf_region_condition_inventory"]
atf_counts = results["atf_region_condition_counts"]
data_contract = results["data_source_contract"]

print("written outputs:", sorted((PROJECT_ROOT / "outputs" / "provenance").glob("*.csv")))
print("db rows:", len(db_summary), "trace rows:", len(trace_summary), "ATF rows:", len(atf_inventory))

## Dataset contract: legacy historical assets versus new ATF traces

In [ ]:
display(data_contract)

## Historical DB study inventory

In [ ]:
display(db_summary)

fig, ax = plt.subplots(figsize=(10, 4))
for condition, group in db_summary.groupby("condition"):
    ax.plot(group["current_na"], group["best_objective"], marker="o", label=condition)
ax.set_xlabel("Current (nA)")
ax.set_ylabel("Best historical objective")
ax.set_title("Historical best objective by condition and current")
ax.legend()
plt.show()

## Historical trace sources used by the legacy objectives

In [ ]:
display(trace_summary)

## New ATF inventory and region × condition counts

In [ ]:
display(atf_counts)
display(atf_inventory)

## Historical objective reproducibility against documented trace sources

In [ ]:
summary = (
    provenance.groupby(["condition", "chosen_status"], dropna=False)
    .size()
    .rename("n_dbs")
    .reset_index()
    .sort_values(["condition", "chosen_status"])
    .reset_index(drop=True)
)
display(summary)
display(provenance)

## Consequence for later steps

The step 00 result is now cleaner and more honest:

- **BARIUM** historical objectives remain reproducible from `BARIUM_TRACES.csv`.
- **MFA** is mostly reproducible, with `MFA_150nA` still unresolved.
- **CONTROL** remains unresolved under the documented `CONTROL_TRACES.csv` source.

That means step 01 should be interpreted as a **legacy diagnostic step**, while step 02 remains the main immediate reviewer-facing gain because it rebuilds thresholds directly from the 37 ATF files.